# 08. K-Means Clustering

## What is this notebook about?

**K-Means** is the most popular clustering algorithm. Given a number K, it partitions your data into K groups by:

1. Picking K random "centroids".
2. Assigning each data point to its nearest centroid.
3. Moving each centroid to the mean of its assigned points.
4. Repeating until nothing changes.

The result: K clean groups, with no labels needed.

## What you will learn

1. How K-Means partitions a synthetic dataset
2. How to **pick the right K** with the **elbow method** and **silhouette score**
3. How to apply K-Means to a real customer-segmentation problem
4. Why **scaling matters** before K-Means

## Datasets

- Synthetic blobs (so you can SEE the clusters)
- Mall Customers from Kaggle (loaded from public URL, no Kaggle account needed)

In [ ]:
# If you're running locally, uncomment and run this once.
# In Google Colab, all of these are pre-installed - you can skip this cell.
# !pip install -q numpy pandas scikit-learn matplotlib seaborn scipy xgboost lightgbm

In [ ]:
# Standard imports used in every ML notebook
import numpy as np                 # numerical arrays
import pandas as pd                # tabular data (DataFrames)
import matplotlib.pyplot as plt    # plotting
import seaborn as sns              # prettier statistical plots

# Set up plot style
sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (10, 6)

# Set a random seed so results are reproducible
np.random.seed(42)

In [ ]:
from sklearn.datasets import make_blobs
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score

## Step 1. Synthetic blobs - the easy case

Make 4 clearly separated groups so we can see exactly what K-Means does.

In [ ]:
# Create 400 points in 4 clusters
X, _ = make_blobs(n_samples=400, centers=4, cluster_std=0.8, random_state=42)

# Plot the raw points (no labels yet)
plt.figure(figsize=(8, 6))
plt.scatter(X[:, 0], X[:, 1], s=15, color="gray")
plt.title("Raw data (we don't know the groups yet)")
plt.show()

You can see 4 visual clumps. But the algorithm doesn't get to see this picture - it only sees coordinates.

In [ ]:
# Run K-Means with K=4
# n_init=10 means try 10 random starts and keep the best
km = KMeans(n_clusters=4, n_init=10, random_state=42).fit(X)

# Plot points colored by cluster, with red X marks at the centroids
plt.figure(figsize=(8, 6))
plt.scatter(X[:, 0], X[:, 1], c=km.labels_, s=15, cmap="tab10")
plt.scatter(km.cluster_centers_[:, 0], km.cluster_centers_[:, 1],
            s=200, marker="X", c="red", label="centroids")
plt.title("K-Means clusters (K=4)")
plt.legend()
plt.show()

K-Means correctly recovered the 4 groups. But how would you have known to use K=4 without looking?

## Step 2. Elbow method + silhouette score

If you don't know K up-front, try several values and look at:

1. **Inertia (WCSS):** sum of squared distances from each point to its centroid. Always decreases as K grows. Look for an "elbow" where the decrease slows.
2. **Silhouette score:** measures cluster separation. Range -1 to 1, higher is better.

In [ ]:
ks = range(1, 11)
inertias = []
silhouettes = []

for k in ks:
    km = KMeans(n_clusters=k, n_init=10, random_state=42).fit(X)
    inertias.append(km.inertia_)
    if k > 1:                                    # silhouette undefined for K=1
        silhouettes.append(silhouette_score(X, km.labels_))

# Plot side by side
fig, ax = plt.subplots(1, 2, figsize=(14, 4))

ax[0].plot(ks, inertias, "o-", color="black")
ax[0].set_xlabel("K")
ax[0].set_ylabel("Inertia (WCSS)")
ax[0].set_title("Elbow Method - look for the bend")

ax[1].plot(list(ks)[1:], silhouettes, "o-", color="black")
ax[1].set_xlabel("K")
ax[1].set_ylabel("Silhouette score (higher is better)")
ax[1].set_title("Silhouette Score by K")
plt.show()

**Both plots peak/elbow at K=4** - confirming the right answer. In real-world data this is rarely so clean, but the same techniques apply.

## Step 3. Real data - Mall Customers

The Mall Customers dataset has 200 customers with their annual income and spending score. We want to find natural customer segments.

In [ ]:
# Try to load from a public URL (no Kaggle account needed)
url = "https://raw.githubusercontent.com/SteffiPeTaffy/machineLearningAZ/master/Machine%20Learning%20A-Z%20Template%20Folder/Part%204%20-%20Clustering/Section%2024%20-%20K-Means%20Clustering/Mall_Customers.csv"
try:
    mall = pd.read_csv(url)
    print(f"Loaded {len(mall)} customers")
    mall.head()
except Exception as e:
    print(f"Could not fetch online: {e}")
    print("If this fails, download Mall_Customers.csv manually from Kaggle and load it.")
    mall = None

In [ ]:
if mall is not None:
    # Pick the two most useful features for visualization
    Xm = mall[["Annual Income (k$)", "Spending Score (1-100)"]].values

    # IMPORTANT: scale features! K-Means is distance-based.
    Xm_s = StandardScaler().fit_transform(Xm)

    # Try K=5 (a common number for customer segments)
    km = KMeans(n_clusters=5, n_init=10, random_state=42).fit(Xm_s)

    # Plot in original (unscaled) space for interpretability
    plt.figure(figsize=(10, 6))
    plt.scatter(Xm[:, 0], Xm[:, 1], c=km.labels_, s=40, cmap="tab10")
    plt.xlabel("Annual Income (k$)")
    plt.ylabel("Spending Score (1-100)")
    plt.title("Mall Customer Segments (K=5)")
    plt.show()

**Real-world segments revealed!** You'll typically see:
- High income, high spending (the dream customer)
- High income, low spending (target with marketing!)
- Low income, high spending (enthusiasts to cherish)
- Low income, low spending (value seekers)
- Average everything (the middle)

This is exactly how marketing teams use K-Means.

## Step 4. Exercises

1. **Try different K values** (3, 4, 6, 7) and decide which makes most business sense.

2. **Use all numeric features** of Mall Customers including Age. Does the result change?

3. **Compare `init="random"` vs `"k-means++"`** with `n_init=1`. K-means++ is the smart initialization.

4. **Try clustering Iris** without using the species labels. Does K-Means recover the 3 species?

5. **Try DBSCAN** instead - it doesn't need K up-front and finds arbitrary shapes:
   ```python
   from sklearn.cluster import DBSCAN
   ```

Next: **09 - Hierarchical Clustering**!